# Aqui calcularemos o RWA do portfolio ficticio

Cálculo do RWA (Risk-Weighted Assets) para Crédito.

O RWA define quanto de capital o banco precisa manter para cada real emprestado.

- A Regra (Resolução 4.958): Ativos diferentes recebem pesos de risco (RW) diferentes.

- No Python: Criaremos um mapeamento de Produto para RW.

 Exemplo: Empréstimo sem garantia (100%), Parcelado Loja (75% ou 100% conforme o caso).

- Fórmula: $RWA = Valor\_Nocional \times RW$.

- Métrica Final: Calcule o Índice de Basileia teórico, dividindo um Patrimônio de Referência (PR) fictício pelo RWA total.

## Diferencas entre bancos S1 e S3

### Capital RWA e basileia
- **Bancos S1** utilizam com frequencia modelos internos avancados IRB (internal Ratings-based) para credito e IMA (internal models approach) para mercados

    - O banco estima suas proprias metricas de PD, LGD e EAD atraves de modelos estatisticos pesados para calcular o RWA. **Dependendo muito da qualidade dos dados.**

- **Bancos S3** utilizam a abordagem padrionizada. o banco central definie tabelas fixas de pesos de risco RW. por exemplo, uma exposicao de var3ejo padrao tem peso de 75% ou 100%

    - Impacto o calculo e mais simples de implementar, mas oferece margem para otimizacao do que os modelos internos dos S1

### Risco de taxa de juros (IRRBB)

A forma como o banco mede impact dos juros no **patrimonio** (EVE) e no **lucro** (NII) muda drasticamente

- **Bancos S1** Realizam calculos de forma granular, simulando choques em cada vertice da curva de juros e utilizando modelos comportamentais dinamicos para cada produto

- **Bancos S3** Seguem a abordagem padronizada do IRRBB (SA-IRRBB) definida na regulamentacao do BCB. Calculo: o banco agrupa os ativos e passivos em janelas de tempo (time buckets) pre-definidas. O choque de juros e aplicado sobre esses bl;ocos de forma mais simplificada

### Provisonamento (Resolução 2.682 vs. 4.966)

Estamos hoje, 07/03/2026, em um momento de transicao importante.

- Regra atual (ambos S1 e S3): ambos seguem a logica de perda incorrida (atraso gera provisao de AA a H)

- Diferenca na gestao: no S1 a gestao de risco e proativa, com modelos de provisioning rodando em tempo real. No S3  o foco e a precisao do reporte regulatorio (CADOC 3040 - responsavel pelo risco de credito)

- **Futuro (IRFS 9/ Res. 4.966)** para os bancos S1 teremos o modelo da **perda esperada**. O S3 tem cronogramas e exigencias mais simplificadas para essa transicao, mas ainda assim precisa implementar modelos de PD e LGD.

## Pesos de RW definidos pelo Bacen para o calculo do RWACAD (ativos ponderados pelo risco de credito sob abordagem padronizada)

Baseados na res. CMN 4958/2021 e Norm. BCB 200/2021

### Pesos de riscos - RWACAD

Fontes Oficiais Utilizadas:

Resolução CMN nº 4.958, de 21 de outubro de 2021: Estabelece os requisitos mínimos de Patrimônio de Referência (PR), de Nível I e de Capital Principal para as instituições do Segmento 3 (S3).

Instrução Normativa BCB nº 200, de 1º de dezembro de 2021: Estabelece os procedimentos para o cálculo da parcela dos ativos ponderados pelo risco (RWA) relativa às exposições ao risco de crédito sujeitas ao cálculo do capital pelo enfoque padronizado (RWACAD).

Resolução CMN nº 4.557, de 23 de fevereiro de 2017: Dispõe sobre a estrutura de gerenciamento de riscos e a estrutura de gerenciamento de capital (GARC).

| Tipos de exposicao/produto | Peso de risco (RW/FPR) | Requisito/ fonte oficial |
| :--- | :---: | ---: |
| `Exposicao de varejo qualificadas` | 75% | Devem atender criterios de granularidade e limite de valor por contraparte |
| `Exposicao de varejo nao qualificadas` | 100% | Exposicoes que nao atendem aos criterios de diversificacao da norma |
| `Crédito Imobiliário (Residencial)` | 35% a 100% | Varia conforme a relação Loan-to-Value (LTV) e o método de avaliação. |
| `Operações Vencidas (> 90 dias)` | 100% ou 150% | 150% se a provisão for < 20%; 100% se a provisão for ≥ 20% do valor da exposição. |
| `Títulos Públicos Federais (Brasil)` | 0% | Exposições ao Governo Federal ou Banco Central do Brasil em moeda nacional. |
| `Exposições a Instituições Financeiras` | 20% a 150% | Depende da nota de crédito (rating) ou da classificação de risco do país de origem.|


Obs: Para bancos S3 e uma obrigacao o usu da abordagem padronizada

In [1]:
# importacao de bibliotecas

import pandas as pd
import numpy as np

from collections import defaultdict

In [2]:
# leitura da base ja passado nos testes de data quality no processo data_analisys_portfolio

df_base = pd.read_csv(r'/home/ricardo/Documents/Projects/analise_alm/data/base_clientes_varejo_simulada.csv')
df_base.head(5)

,ID_Contrato,Produto,Valor_Nocional,Taxa_Contratada_AA,Prazo_Remanescente_Meses,Indexador,Rating_2682,Dias_Atraso
0,1001,Parcelado_Loja,23774.04,0.2138,42,POS-CDI,AA,0
1,1002,Emprestimo_Pessoal,37341.81,0.1830,21,PRE,AA,0
2,1003,Parcelado_Loja,43655.72,0.2419,35,PRE,A,0
3,1004,Parcelado_Loja,21965.94,0.2565,41,PRE,A,0
4,1005,Emprestimo_Pessoal,2661.25,0.1226,40,POS-CDI,AA,0


In [3]:
# calculando a provisao vl_provisao = nocional * % rating

# a provisao e a reserva para a perda esperada. Vamos criar um dicionario de mapeamento 
# onde cada letra corresponde a um percentual fixo definido pelo BCB.

# dict de provisao Resolução CMN nº 2.682
dict_provisao = {
    'AA': 0
    ,'A': 0.005
    ,'B': 0.01
    ,'C': 0.03
    ,'D': 0.10
    ,'E': 0.30
    ,'F': 0.50
    ,'G': 0.70
    ,'H': 1
}

df_analise = df_base.copy()


df_analise['Provisao'] = df_analise['Valor_Nocional'] * df_analise['Rating_2682'].apply(lambda x: dict_provisao[x])
df_analise.head(5)


,ID_Contrato,Produto,Valor_Nocional,Taxa_Contratada_AA,Prazo_Remanescente_Meses,Indexador,Rating_2682,Dias_Atraso,Provisao
0,1001,Parcelado_Loja,23774.04,0.2138,42,POS-CDI,AA,0,0.0000
1,1002,Emprestimo_Pessoal,37341.81,0.1830,21,PRE,AA,0,0.0000
2,1003,Parcelado_Loja,43655.72,0.2419,35,PRE,A,0,218.2786
3,1004,Parcelado_Loja,21965.94,0.2565,41,PRE,A,0,109.8297
4,1005,Emprestimo_Pessoal,2661.25,0.1226,40,POS-CDI,AA,0,0.0000


In [4]:
# atribuicao de pesos de risco, RW, - IN BCB 200/2021

df_analise['Produto'].unique()

<StringArray>
['Parcelado_Loja', 'Emprestimo_Pessoal', 'Cartao_Credito']
Length: 3, dtype: str

In [5]:
# criar dict para os produtos da base que estamos analisando

prod = defaultdict(str)

# adicionar produtos
prod['Parcelado_Loja'] = 'Varejo qualificado'
prod['Emprestimo_Pessoal'] = 'Varejo qualificado'
prod['Cartao_Credito'] = 'Varejo qualificado'

prod

defaultdict(str,
            {'Parcelado_Loja': 'Varejo qualificado',
             'Emprestimo_Pessoal': 'Varejo qualificado',
             'Cartao_Credito': 'Varejo qualificado'})

In [8]:
# dias de atraso operacao vencida > 90 dias

def cal_rw(linha):

    dias_atraso = linha['Dias_Atraso']
    provisao = linha['Provisao']
    exposicao = linha['Valor_Nocional']
    produto = linha['Produto']

    if dias_atraso > 90:

        if provisao >= (0.20 * exposicao):
            return 1
        else:
            return 1.50
        
    else:

        prodt_dict = prod[produto]
        if prodt_dict == 'Varejo qualificado':
            return 0.75
        else:
            return None
        

df_analise['rw'] = df_analise.apply(cal_rw, axis = 1)

df_analise

,ID_Contrato,Produto,Valor_Nocional,Taxa_Contratada_AA,Prazo_Remanescente_Meses,Indexador,Rating_2682,Dias_Atraso,Provisao,rw
0,1001,Parcelado_Loja,23774.04,0.2138,42,POS-CDI,AA,0,0.00000,0.75
1,1002,Emprestimo_Pessoal,37341.81,0.1830,21,PRE,AA,0,0.00000,0.75
2,1003,Parcelado_Loja,43655.72,0.2419,35,PRE,A,0,218.27860,0.75
3,1004,Parcelado_Loja,21965.94,0.2565,41,PRE,A,0,109.82970,0.75
4,1005,Emprestimo_Pessoal,2661.25,0.1226,40,POS-CDI,AA,0,0.00000,0.75
...,...,...,...,...,...,...,...,...,...,...
1995,2996,Emprestimo_Pessoal,5632.09,0.1784,14,PRE,A,0,28.16045,0.75
1996,2997,Parcelado_Loja,2908.55,0.2914,36,PRE,A,0,14.54275,0.75
1997,2998,Cartao_Credito,29260.25,0.2683,5,PRE,AA,0,0.00000,0.75
1998,2999,Parcelado_Loja,21590.51,0.2476,11,PRE,B,0,215.90510,0.75


In [9]:
# calculo do RWA

df_analise['rwa'] = df_analise['Valor_Nocional'] * df_analise['rw']

df_analise

,ID_Contrato,Produto,Valor_Nocional,Taxa_Contratada_AA,Prazo_Remanescente_Meses,Indexador,Rating_2682,Dias_Atraso,Provisao,rw,rwa
0,1001,Parcelado_Loja,23774.04,0.2138,42,POS-CDI,AA,0,0.00000,0.75,17830.5300
1,1002,Emprestimo_Pessoal,37341.81,0.1830,21,PRE,AA,0,0.00000,0.75,28006.3575
2,1003,Parcelado_Loja,43655.72,0.2419,35,PRE,A,0,218.27860,0.75,32741.7900
3,1004,Parcelado_Loja,21965.94,0.2565,41,PRE,A,0,109.82970,0.75,16474.4550
4,1005,Emprestimo_Pessoal,2661.25,0.1226,40,POS-CDI,AA,0,0.00000,0.75,1995.9375
...,...,...,...,...,...,...,...,...,...,...,...
1995,2996,Emprestimo_Pessoal,5632.09,0.1784,14,PRE,A,0,28.16045,0.75,4224.0675
1996,2997,Parcelado_Loja,2908.55,0.2914,36,PRE,A,0,14.54275,0.75,2181.4125
1997,2998,Cartao_Credito,29260.25,0.2683,5,PRE,AA,0,0.00000,0.75,21945.1875
1998,2999,Parcelado_Loja,21590.51,0.2476,11,PRE,B,0,215.90510,0.75,16192.8825


In [10]:
# capital minimo para S3

cap_min = df_analise['rwa'].sum() * 0.08
cap_min

np.float64(3090314.5074)

# Calculo do RWA para S1

Diferente do S3 onde o peso RW e fixo por tabelas no S1 o risco e uma funcao continua e nao olinear baseada em probabilidades.

A regulamentacao brasileira que rege os modelos internos de credito esta consolidada na Res. BCB 303/2023 (atualizando a 3648)

## 1. Os pilares do modelo IRB (S1)

Diferente da abordagem padronizada, voce precisa estimar internamente tres parametros fundamentais para cada contrato ou segmento

- **PD (probabilidade de default)**: prob de um cliente dar default em 1 ano;

- **LGD (Loss Given Default)**: Percentual da exposicao que o banco efetivamente perde apos o calote;

- **EAD (Exposure at Default)**: o valor da exposicao no momento do default.

## 2. A formula de ponderacao de risco (Varejo)

Para a esxposicao de varejo (como cartoes e emprestimos), o banco cenral define a correlacao, R, e o requisito de capital, K, atraves da formula baseada na distribuicao normal.

- Calculo da correlacao R

$$R = 0,03 \cdot \left( \frac{1 - e^{-35 \cdot PD}}{1 - e^{-35}} \right) + 0,16 \cdot \left[ 1 - \left( \frac{1 - e^{-35 \cdot PD}}{1 - e^{-35}} \right) \right]$$

- calculo do capital K

$$K = \left[ LGD \cdot N \left( \frac{G(PD)}{\sqrt{1 - R}} + \sqrt{\frac{R}{1 - R}} \cdot G(0,999) \right) - (PD \cdot LGD) \right]$$

onde: 
    - N(x) e a funcao de distribuicao normal acumulada
    - G(z) e a inversa da funcao de distribuicao normal acumulada
    - RWA final: RWA = K * 12.5 * EAD

## 3. Regra do output floor (S1 vs S3)

Uma regra crucial para bancos S1 e o limite de otimizacao de capital. O comite de basileia III  esabelece que o RWA calculado por modelos internos nao pode ser inferior a um percentual do valor que seria obtido pelo modelo padronizado

- **Piso atual**: O RWA via IRB deve ser no minimo 72.5% do RWA que o banco calcularia se usasse a abordagem padronizada (S3)

- **Objetivo**: Evitar que bancos grandes reduzam excessivamente o capital regulatorio atraves de modelos internos agressivos

### Para que possamos calcular o RWA para um banco S1 precisamos de informacoes extras

 - PD
 - LGD
 - EAD